In [4]:
import multiprocessing
import os , json , requests
import re
import nltk
import gensim.models.word2vec as w2v
import sklearn.manifold
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.contrib.tensorboard.plugins import projector

In [2]:
!pip install nltk

  Created wheel for nltk: filename=nltk-3.4.5-py3-none-any.whl size=1449907 sha256=77e04a1febab8a4626995c290d8f13a910a0dd16eb34ccbef97fb4ba112a2ec9
  Stored in directory: c:\users\admin\appdata\local\pip\cache\wheels\e3\c9\b0\ed26a73ef75a53145820825afa8e2d2c9b30fe9f6c10cd3202
Successfully built nltk


In [5]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [17]:
"""## Prepare Corpus

**Load books from url**
"""
def sentence_to_wordlist(raw):
    clean=re.sub("[^a-zA-Z]", " ", raw)
    words = clean.split()
    return map(lambda x:x.lower(), words)

filepath = 'http://www.gutenberg.org/files/33224/33224-0.txt'

corpus_raw = requests.get(filepath).text

tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

raw_sentences = tokenizer.tokenize(corpus_raw)

#sentence where each word is tokenized
sentences = []
for raw_sentence in raw_sentences:
    if len(raw_sentence) > 0:
        sentences.append(list(sentence_to_wordlist(raw_sentence)))

token_count = sum([len(sentence) for sentence in sentences])
print("The book corpus contains {} tokens".format(token_count))

The book corpus contains 425633 tokens


In [14]:
sentences = list(sentences)

In [18]:
 """## Train Word2Vec"""
    
# Dimensionality of the resulting word vectors more dimensions, more computationally expensive to train but also more accurate
# more dimensions = more generalized

num_features = 300

# Minimum word count threshold
min_word_count = 3

# more workers, faster wer train
num_workers = multiprocessing.cpu_count()

# Context window length
context_size = 7

# Downsample setting for frequent words
downsampling = 1e-3

seed = 1

client2vec = w2v.Word2Vec(sg=1,
                         seed=seed,
                         workers=num_workers, 
                         size=num_features, 
                         min_count=min_word_count, 
                         window=context_size, 
                         sample=downsampling)

client2vec.build_vocab(sentences)

client2vec.corpus_count

18226

In [19]:
"""**Start training, this might take a minute or two...**"""

client2vec.train(sentences ,total_examples=client2vec.corpus_count , epochs=10)

"""**Save to file, can be useful later**"""

if not os.path.exists(os.path.join("trained",'sample')):
    os.makedirs(os.path.join("trained",'sample'))

client2vec.save(os.path.join("trained",'sample', ".w2v"))

In [20]:
client2vec.most_similar("earth")

D:\Library\Anaconda3\envs\ocr_env\lib\site-packages\ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  """Entry point for launching an IPython kernel.


[('crust', 0.7178483009338379),
 ('globe', 0.6683126091957092),
 ('planet', 0.6052641868591309),
 ('inequalities', 0.5985287427902222),
 ('moon', 0.590129017829895),
 ('sun', 0.5850935578346252),
 ('unevenness', 0.5847407579421997),
 ('laboring', 0.5801404714584351),
 ('orbit', 0.5789523124694824),
 ('remodelled', 0.5616137981414795)]

In [21]:
client2vec.most_similar_cosmul(positive=['moon','earth'], negative=['orbit'])

D:\Library\Anaconda3\envs\ocr_env\lib\site-packages\ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `most_similar_cosmul` (Method will be removed in 4.0.0, use self.wv.most_similar_cosmul() instead).
  """Entry point for launching an IPython kernel.


[('crust', 0.8318690657615662),
 ('remodelled', 0.8064056038856506),
 ('sun', 0.806380033493042),
 ('god', 0.8001940250396729),
 ('autobiography', 0.7834774255752563),
 ('globe', 0.7824045419692993),
 ('planet', 0.7813032269477844),
 ('outwards', 0.7767391800880432),
 ('hutton', 0.776357889175415),
 ('heaven', 0.7761793732643127)]

In [ ]:
tsne = sklearn.manifold.TSNE(n_components=2, random_state=0)

all_word_vectors_matrix = client2vec.wv.vectors




all_word_vectors_matrix_2d = tsne.fit_transform(all_word_vectors_matrix)

points = pd.DataFrame(
    [
        (word, coords[0], coords[1])
        for word, coords in [
            (word, all_word_vectors_matrix_2d[client2vec.wv.vocab[word].index])
            for word in client2vec.wv.vocab
        ]
    ],
    columns=["word", "x", "y"]
)

In [ ]:
points.head(10)

In [ ]:
sns.set_context("poster")
    
ax = points.plot.scatter("x", "y", s=10, figsize=(20, 12))


fig = ax.get_figure()
fig.savefig(os.path.join("trained",'sample'+".png"))

In [ ]:
def plot_region(x_bounds, y_bounds):
        slice = points[
            (x_bounds[0] <= points.x) &
            (points.x <= x_bounds[1]) & 
            (y_bounds[0] <= points.y) &
            (points.y <= y_bounds[1])
        ]
        
        ax = slice.plot.scatter("x", "y", s=35, figsize=(10, 8))
        for i, point in slice.iterrows():
            ax.text(point.x + 0.005, point.y + 0.005, point.word, fontsize=11)

In [ ]:
plot_region(x_bounds=(40.0, 60.0), y_bounds=(-40.0, -32.0))

In [ ]:
client2vec.predict_output_word(['earth','orbit'])

In [ ]:
vocab_list = points.word.values.tolist()
embeddings = all_word_vectors_matrix


embedding_var = tf.Variable(all_word_vectors_matrix, dtype='float32', name='embedding')
projector_config = projector.ProjectorConfig()


embedding = projector_config.embeddings.add()
embedding.tensor_name = embedding_var.name

LOG_DIR='./'
metadata_file = os.path.join("sample.tsv")



with open(os.path.join(LOG_DIR, metadata_file), 'wt') as metadata:
    metadata.writelines("%s\n" % w.encode('utf-8') for w in vocab_list)

embedding.metadata_path =  os.path.join(os.getcwd(), metadata_file)

# Use the same LOG_DIR where you stored your checkpoint.
summary_writer = tf.summary.FileWriter(LOG_DIR)

# The next line writes a projector_config.pbtxt in the LOG_DIR. TensorBoard will
# read this file during startup.
projector.visualize_embeddings(summary_writer, projector_config)

saver = tf.train.Saver([embedding_var])

with tf.Session() as sess:
    # Initialize the model
    sess.run(tf.global_variables_initializer())

    saver.save(sess, os.path.join(LOG_DIR, metadata_file+'.ckpt'))

In [ ]:
LOG_DIR = './'
get_ipython().system_raw(
    'tensorboard --logdir {} --host 0.0.0.0 --port 6006 &'
    .format(LOG_DIR)
)

get_ipython().system_raw('lt --port 6006 >> url.txt 2>&1 &')
! cat url.txt